In [1]:
print("okay")

okay


In [2]:
%pwd

'g:\\Langchain_projects\\Medi-Ai-flask\\Medi-AI\\research'

In [7]:
import os
os.chdir("g:\Langchain_projects\Medi-Ai-flask\Medi-AI")
%pwd

'g:\\Langchain_projects\\Medi-Ai-flask\\Medi-AI'

In [8]:
from langchain.document_loaders import PyPDFLoader,DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [11]:
def load_pdf_files(data):
    loader= DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents= loader.load()
    return documents

In [12]:
extracted_data= load_pdf_files("data")

In [13]:
len(extracted_data)

809

In [14]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [15]:
minimal_docs= filter_to_minimal_docs(extracted_data)

In [16]:
minimal_docs

[Document(metadata={'source': 'data\\Textbook of Biochemistry for Medical Students - PDF Room.pdf'}, page_content=''),
 Document(metadata={'source': 'data\\Textbook of Biochemistry for Medical Students - PDF Room.pdf'}, page_content='Textbook of\nBIOCHEMISTRY'),
 Document(metadata={'source': 'data\\Textbook of Biochemistry for Medical Students - PDF Room.pdf'}, page_content=''),
 Document(metadata={'source': 'data\\Textbook of Biochemistry for Medical Students - PDF Room.pdf'}, page_content='DM Vasudevan MBBS\xa0MD\xa0FAMS FRCPath\nDistinguished Professor\nDepartment of Biochemistry \nCollege of Medicine, Amrita Institute of Medical Sciences\nKochi, Kerala, India\nFormerly\nPrincipal, College of Medicine\nAmrita Institute of Medical Sciences, Kerala, India\nDean, Sikkim Manipal Institute of Medical Sciences\nGangtok, Sikkim, India\nSreekumari S MBBS MD\nProfessor and Head\nDepartment of Biochemistry\nSree Gokulam Medical College and Research Foundation\nThiruvananthapuram, Kerala, Indi

In [20]:
#split the documents
def text_split(minimal_docs):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,

    )
    texts_chunk=text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [21]:
text_chunk= text_split(minimal_docs)
print(f"number of chunks:{len(text_chunk)}")

number of chunks:6074


In [23]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and return the HuggingFace embeddings model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings

embedding = download_embeddings()

C:\Users\Sahil goyal\AppData\Local\Temp\ipykernel_3712\30168171.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
g:\Langchain_projects\Medi-Ai-flask\Medi-AI\medibot\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sahil goyal\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE

In [24]:
vector = embedding.embed_query("Hello world")
vector

[-0.034477267414331436,
 0.031023245304822922,
 0.006735002622008324,
 0.02610897272825241,
 -0.03936199098825455,
 -0.16030247509479523,
 0.06692401319742203,
 -0.006441466510295868,
 -0.047450482845306396,
 0.014758829958736897,
 0.07087530195713043,
 0.05552758648991585,
 0.01919335499405861,
 -0.026251377537846565,
 -0.010109559632837772,
 -0.02694047801196575,
 0.022307416424155235,
 -0.02222663350403309,
 -0.1496925950050354,
 -0.01749303564429283,
 0.0076762172393500805,
 0.054352302104234695,
 0.003254439914599061,
 0.0317259281873703,
 -0.0846213549375534,
 -0.0294059868901968,
 0.051595572382211685,
 0.048124056309461594,
 -0.003314792178571224,
 -0.058279186487197876,
 0.04196929186582565,
 0.022210659459233284,
 0.128188818693161,
 -0.022338882088661194,
 -0.011656327173113823,
 0.06292837858200073,
 -0.03287631645798683,
 -0.0912260040640831,
 -0.031175334006547928,
 0.052699558436870575,
 0.04703483358025551,
 -0.08420311659574509,
 -0.0300561785697937,
 -0.02074487134814

In [66]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [67]:
PINECONE_API_KEY=os.getenv("PINECONE_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")



os.environ["PINECONE_API_KEY"]= PINECONE_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [40]:
from pinecone import Pinecone
Pinecone_api_key=PINECONE_API_KEY

pc= Pinecone(api_key=Pinecone_api_key)

In [44]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index = pc.Index(index_name)

In [45]:
from langchain_pinecone import PineconeVectorStore

docsearch= PineconeVectorStore.from_documents(
    documents=text_chunk,
    embedding=embedding,
    index_name=index_name)

In [52]:
retriever = docsearch.as_retriever(search_type="similarity",search_kwargs={"k":3})

In [68]:
retrieved_docs=retriever.invoke("what is the Reaction of Amide group")
retrieved_docs

[Document(id='835bcb58-ab3b-42ae-89e0-ed77e46fed1c', metadata={'source': 'data\\Textbook of Biochemistry for Medical Students - PDF Room.pdf'}, page_content='30 Textbook of Biochemistry\nAmide Formation\nThe-COOH\tgroup \tof \tdicarboxylic \tamino \tacids \t(other \tthan \t\nalpha carboxyl) can combine with ammonia to form the \ncorresponding amide. For example, \n\t Aspartic \tacid\t+\tNH3  → Asparagine\n\t Glutamic \tacid\t+\tNH3 →\t Glutamine \t(Fig.3.19)\nThese\tamides \tare \talso \tcomponents \tof \tprotein \tstructure. \tThe \t\namide group of glutamine serves as the source of nitrogen \nfor nucleic acid synthesis.\nDue to Amino Group\nTransamination'),
 Document(id='7d0e148c-107e-4f58-9ba8-3a3b21577ef2', metadata={'source': 'data\\Textbook of Biochemistry for Medical Students - PDF Room.pdf'}, page_content='Similarly,\tthese \thydroxyl \tgroups \tcan \tform \tO-glycosidic \t\nbonds with carbohydrate residues to form glycoproteins.\nReaction of the Amide Group\nThe\tamide \tgrou

In [70]:
from langchain_groq import ChatGroq

chatModel = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [55]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [73]:
system_prompt = (
    "You are an expert Biochemistry tutor for medical students. Answer questions using only the provided textbook context. Explain concepts clearly and accurately in simple language. If the answer is not available in the context, say you don't have enough information instead of guessing. Use bullet points when appropriate and include clinical relevance if mentioned in the context."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

response = rag_chain.invoke({"input": " 'Any Help In Learning These Little Molecules Proves Truely Valuable' this phrase stands for?"})
print(response["answer"])

The phrase "Any Help In Learning These Little Molecules Proves Truely Valuable" seems to be an acronym that stands for the word "AHLTLMPTV" but when expanded with the first letter of each word, it forms the phrase. However, I couldn't find any direct reference to this phrase in the provided textbook context.

But if we consider the general context of biochemistry, it seems like the phrase is referring to the importance of learning about small molecules or biomolecules in biochemistry. If I had to guess, I'd say it might be related to the phrase "Any Help In Learning These Little Molecules Proves Truly Valuable" = Amino acids, Hormones, Lipids, Sugars, Nucleotides, etc. which are all small molecules or biomolecules, but I don't have enough information to confirm this.
